In [1]:
import torch
from torch.utils import data
from collections import Counter
from torch import nn
epoch=300
batch_size = 64
num_steps = 20
lr=0.1

data_path = "../data/fra-eng/fra.txt"
device=torch.device('cuda')

In [2]:
def data_read():
    with open(data_path,'r') as file:
        text=file.read()
    return text

In [3]:
def preprocess_nmt(text):
    text = (
        text
        .replace("\u202f", " ")
        .replace("\xa0", " ")
        .lower()
    )
    result=[]
    for i,char in enumerate(text):
        if (
            char in ",.!?"
            and i > 0
            and text[i - 1] != " "
        ):
            result.append(' ')
        result.append(char)
    return "".join(result)

In [4]:
def tokenize_nmt(text,max_text=600):
    source=[]
    target=[]
    for line in text.splitlines():
        parts=line.split('\t')

        if len(parts)<2:
            continue
        source.append(parts[0].split())
        target.append(parts[1].split())
        if(max_text is not None and len(source)>=max_text):
            break
    return source,target

In [5]:
def vocab_maker(tokens,min_freq=2):
    counter=Counter()
    for line in tokens:
        counter.update(line)
    token_freq=sorted(counter.items(),key=lambda x:(-x[1],x[0]))
    idx_to_token=["<unk>",
        "<pad>",
        "<bos>",
        "<eos>"]
    for token,freq in token_freq:
        if freq<min_freq :
            continue
        if token not in idx_to_token:
            idx_to_token.append(token)
    token_to_idx={
        token:idx for idx,token in enumerate(idx_to_token)
    }
    vocab = {
        "idx_to_token": idx_to_token,
        "token_to_idx": token_to_idx
    }

    return vocab

In [6]:
def truncate_pad(line,num_step,pad_idx):
    if len(line)>num_step:
        return line[:num_step]
    return line+[pad_idx]*(num_step-len(line))


In [7]:
def build_array_nmt(lines, vocab, num_steps):
    token_to_idx = vocab["token_to_idx"]

    unk_idx = token_to_idx["<unk>"]
    pad_idx = token_to_idx["<pad>"]
    eos_idx = token_to_idx["<eos>"]

    array = []

    for line in lines:
        line_idx = [
            token_to_idx.get(token, unk_idx)
            for token in line
        ]

        # 每句话末尾添加结束标记
        line_idx.append(eos_idx)

        line_idx = truncate_pad(
            line_idx,
            num_steps,
            pad_idx
        )

        array.append(line_idx)

    array = torch.tensor(
        array,
        dtype=torch.long
    )

    valid_len = (
        array != pad_idx
    ).sum(dim=1)

    return array, valid_len

In [8]:
def load_data_nmt(
    batch_size,
    num_steps,
    num_examples=600
):
    raw_text = data_read()

    text = preprocess_nmt(raw_text)

    source, target = tokenize_nmt(
        text,
        num_examples
    )

    src_vocab = vocab_maker(
        source,
        min_freq=2
    )

    tgt_vocab = vocab_maker(
        target,
        min_freq=2
    )

    src_array, src_valid_len = build_array_nmt(
        source,
        src_vocab,
        num_steps
    )

    tgt_array, tgt_valid_len = build_array_nmt(
        target,
        tgt_vocab,
        num_steps
    )

    dataset = data.TensorDataset(
        src_array,
        src_valid_len,
        tgt_array,
        tgt_valid_len
    )

    train_iter = data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    return train_iter, src_vocab, tgt_vocab

In [9]:
train_iter,src_vocab,tgt_vocab=load_data_nmt(batch_size,num_steps)


最开始的x是（batch_size,step）然后经过encode得到output=(step,batch_size,hidden×2)，以及state=(2,batch,hidden_layer)。

接着就只取出state。将dim=0拆开，合并成（batch_size,hidden×2）。接着经过一个线性层变成（batch,hidden）.再unsqueeze成（1，batch，hidden）。这个作为decoder的初始state。

将输入的原始y，最后一个时间步截掉。然后在y的前边拼一个bos。相当于将原本的y里的每个数位置往右移动一位，空出来的位置塞入bos的值。然后作为decoder的输入。目的是读一个输出一个

In [ ]:
class encoder(nn.Module):
    def __init__(self,input_num,embedding_dim,hidden_layer,pad_idx):
        super().__init__()
        self.input_num=input_num
        self.hidden_layer=hidden_layer
        self.embedding=nn.Embedding(num_embeddings=input_num,embedding_dim=embedding_dim,padding_idx=pad_idx)
        self.gru=nn.GRU(embedding_dim,hidden_layer,bidirectional=True)
    def forward(self,x):
        x=self.embedding(x)
        x=x.permute(1,0,2)
        output,state=self.gru(x)
        return output,state

In [11]:
class decoder(nn.Module):
    def __init__(self,input_num,embedding_num,hidden_layer,pad_idx):
        super().__init__()
        self.input_num=input_num
        self.embedding_num=embedding_num
        self.hidden_layer=hidden_layer
        self.embedding=nn.Embedding(num_embeddings=input_num,embedding_dim=embedding_num,padding_idx=pad_idx)
        self.gru=nn.GRU(embedding_num,hidden_layer)
        self.output=nn.Linear(hidden_layer,input_num)
    def forward(self,x,state):
        x=self.embedding(x)
        x=x.permute(1,0,2)
        output,state=self.gru(x,state)
        output=output.reshape(-1,self.hidden_layer)
        output=self.output(output)
        return output,state

In [12]:
class seq2seq(nn.Module):
    def __init__(self,hidden_layer,encoder,decoder,bos_idx):
        super().__init__()
        self.encoder=encoder
        self.decoder=decoder
        self.linear=nn.Linear(hidden_layer*2,hidden_layer)
        #self.input_num=input_num
        #self.embedding_num=embedding_num
        #self.hidden_layer=hidden_layer
        self.bos_idx=bos_idx
    def forward(self,x,y,state=None):
        encoder_output,encoder_state=self.encoder(x)
        forward_state = encoder_state[0]
        backward_state = encoder_state[1]
        context = torch.cat(
            (forward_state, backward_state),
            dim=1
        )
        context = torch.tanh(
            self.linear(context)
        )
        decoder_state = context.unsqueeze(0).repeat(
            self.decoder.gru.num_layers,
            1,
            1
        )
        bos = torch.full(
            size=(y.shape[0], 1),
            fill_value=self.bos_idx,
            dtype=torch.long,
            device=y.device
        )
        decoder_x = torch.cat(
            (bos, y[:, :-1]),
            dim=1
        )
        output, decoder_state = self.decoder(
            decoder_x,
            decoder_state
        )
        return output, decoder_state


In [13]:
# 1. 英文pad给Encoder
src_pad_idx = src_vocab["token_to_idx"]["<pad>"]

# 2. 法文pad和bos给Decoder
tgt_pad_idx = tgt_vocab["token_to_idx"]["<pad>"]
bos_idx = tgt_vocab["token_to_idx"]["<bos>"]
my_encoder=encoder(len(src_vocab['token_to_idx']),128,256,src_pad_idx).to(device)
my_decoder=decoder(len(tgt_vocab['token_to_idx']),128,256,tgt_pad_idx).to(device)
net=seq2seq(256,my_encoder,my_decoder,bos_idx).to(device)
trainer=torch.optim.SGD(net.parameters(),lr=lr)
loss = nn.CrossEntropyLoss(
    ignore_index=tgt_pad_idx
)
for i in range(epoch):
    net.train()

    train_loss = 0.0
    train_count = 0.0

    for x, x_len, y, y_len in train_iter:
        x = x.to(device)
        y = y.to(device)

        trainer.zero_grad()

        logit, state = net(x, y)

        target = y.T.reshape(-1)

        l = loss(logit, target)

        l.backward()

        torch.nn.utils.clip_grad_norm_(
            net.parameters(),
            max_norm=1.0
        )

        trainer.step()

        valid_count = y_len.sum().item()

        train_loss += l.item() * valid_count
        train_count += valid_count

    print(
        f"第{i + 1}个epoch，"
        f"loss：{train_loss / train_count:.4f}"
    )


第1个epoch，loss：4.8477
第2个epoch，loss：3.7142
第3个epoch，loss：3.0876
第4个epoch，loss：2.8561
第5个epoch，loss：2.7396
第6个epoch，loss：2.6573
第7个epoch，loss：2.5934
第8个epoch，loss：2.5419
第9个epoch，loss：2.4964
第10个epoch，loss：2.4573
第11个epoch，loss：2.4269
第12个epoch，loss：2.3946
第13个epoch，loss：2.3676
第14个epoch，loss：2.3429
第15个epoch，loss：2.3213
第16个epoch，loss：2.2984
第17个epoch，loss：2.2778
第18个epoch，loss：2.2592
第19个epoch，loss：2.2389
第20个epoch，loss：2.2206
第21个epoch，loss：2.2029
第22个epoch，loss：2.1843
第23个epoch，loss：2.1674
第24个epoch，loss：2.1517
第25个epoch，loss：2.1355
第26个epoch，loss：2.1193
第27个epoch，loss：2.1051
第28个epoch，loss：2.0886
第29个epoch，loss：2.0745
第30个epoch，loss：2.0593
第31个epoch，loss：2.0466
第32个epoch，loss：2.0321
第33个epoch，loss：2.0180
第34个epoch，loss：2.0030
第35个epoch，loss：1.9916
第36个epoch，loss：1.9777
第37个epoch，loss：1.9641
第38个epoch，loss：1.9516
第39个epoch，loss：1.9381
第40个epoch，loss：1.9263
第41个epoch，loss：1.9115
第42个epoch，loss：1.9000
第43个epoch，loss：1.8867
第44个epoch，loss：1.8762
第45个epoch，loss：1.8621
第46个epoch，loss：1.84

In [14]:
def predict_seq2seq(
    text,
    net,
    src_vocab,
    tgt_vocab,
    num_steps,
    device
):
    net.eval()

    # 1. 清洗并分词英文
    text = preprocess_nmt(text)
    tokens = text.split()

    src_token_to_idx = src_vocab["token_to_idx"]
    tgt_token_to_idx = tgt_vocab["token_to_idx"]

    src_unk_idx = src_token_to_idx["<unk>"]
    src_eos_idx = src_token_to_idx["<eos>"]
    src_pad_idx = src_token_to_idx["<pad>"]

    bos_idx = tgt_token_to_idx["<bos>"]
    eos_idx = tgt_token_to_idx["<eos>"]

    # 2. 英文转编号
    x = [
        src_token_to_idx.get(
            token,
            src_unk_idx
        )
        for token in tokens
    ]

    x.append(src_eos_idx)

    x = truncate_pad(
        x,
        num_steps,
        src_pad_idx
    )

    x = torch.tensor(
        x,
        dtype=torch.long,
        device=device
    ).reshape(1, -1)
    # (1, num_steps)

    output_idx = []

    with torch.no_grad():
        # 3. Encoder
        _, encoder_state = net.encoder(x)

        forward_state = encoder_state[0]
        backward_state = encoder_state[1]

        context = torch.cat(
            (forward_state, backward_state),
            dim=1
        )

        context = torch.tanh(
            net.linear(context)
        )

        decoder_state = context.unsqueeze(0).repeat(
            net.decoder.gru.num_layers,
            1,
            1
        )

        # 4. Decoder第一个输入是<bos>
        decoder_x = torch.tensor(
            [[bos_idx]],
            dtype=torch.long,
            device=device
        )

        # 5. 自回归生成
        for _ in range(num_steps):
            logit, decoder_state = net.decoder(
                decoder_x,
                decoder_state
            )

            predict_idx = logit.argmax(
                dim=1
            ).item()

            if predict_idx == eos_idx:
                break

            output_idx.append(predict_idx)

            # 当前预测作为下一步输入
            decoder_x = torch.tensor(
                [[predict_idx]],
                dtype=torch.long,
                device=device
            )

    # 6. 编号还原为法文
    result = [
        tgt_vocab["idx_to_token"][idx]
        for idx in output_idx
    ]

    return " ".join(result)

In [15]:
predict_seq2seq(
    "go.",
    net,
    src_vocab,
    tgt_vocab,
    num_steps,
    device
)

'va !'